# Import

In [ ]:
import importlib
import random

from deap import base, creator, tools

import config.params as params
import config.seeding as seeding
import config.sensors as sensors


def refresh_configuration():
    """Reload configuration modules so notebook cells see source changes."""
    global params, seeding, sensors
    sensors = importlib.reload(sensors)
    params = importlib.reload(params)
    seeding = importlib.reload(seeding)
    return sensors, params, seeding

# Initialization

In [ ]:
# Define the fitness to maximize
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Define the Individual class as a list of Gene objects with an associated fitness.
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

In [ ]:
from custom_toolbox.initialize import initialize
init = importlib.reload(initialize)

# Register the individual generator.
toolbox.register("individual", init.create_individual, creator.Individual)

# Register the population generator (creates a list of individuals).
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# NOTE: Consider Demes
# A deme is a sub-population that is contained in a population.
# https://deap.readthedocs.io/en/master/tutorials/basic/part1.html#demes

# Create initial population.
refresh_configuration()
population = init.create_seeded_population(
    creator.Individual,
    toolbox.individual,
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

In [ ]:
import importlib

from utils import visualization
viz = importlib.reload(visualization)

viz.visualize_population(population, max_display=10)

# Evaluation

In [ ]:
from custom_toolbox.evaluate import evaluate_fitness_mock
evaluate = importlib.reload(evaluate_fitness_mock)

# Register the custom evaluation function.
toolbox.register("evaluates", evaluate.evaluate_individual)

# Selection

In [ ]:
# Tournament selection with tournament size of 3.
toolbox.register("select", tools.selTournament, tournsize=3)

# Mutation

In [ ]:
from custom_toolbox.mutate import mutate
mutate = importlib.reload(mutate)

# Register the custom mutation function.
toolbox.register("mutate", mutate.mutate_sensor_layout)